# VRM + EP-KD on CIFAR-100: ResNet50 → MobileNetV2 (3 seeds, Kaggle GPU T4 x2)

This notebook clones [Soobiwan/epkd-vision](https://github.com/Soobiwan/epkd-vision). It downloads CIFAR-100 and the
pretrained ResNet50 teacher, then trains the MobileNetV2 student with **VRM + EP-KD**, using the paper's EP-KD recipe
(`scripts/cifar100.sh`) with the VRM objective of mvkd.py in place of KD (weak + RandAugment-strong copy of every image and view, batch 64), for seeds 0, 1 and 2.

| | |
|---|---|
| teacher | `ResNet50`. The standard CRD teacher from [RepDistiller](https://github.com/HobbitLong/RepDistiller), test accuracy 79.34%. If that server fails, the notebook uses the mirror in the [mdistiller](https://github.com/megvii-research/mdistiller) release. |
| student | `MobileNetV2`, base learning rate 0.01 (the CRD / DKD setting for this student) |
| method | `--views 1 --view-cache 1 --view-set sal1,sal2,sal3 --view-rotate 1 --passes 3,3,3,1,1 --anchor 1 --milestones 48,144,192,210 --method vrm --bs 64` |
| KD (ours) | `--views 0 --milestones 150,180,210`, the same seeds (EP-KD notebook only). Off by default (`RUN_KD = False`); set it to `True` for the "KD (ours)" row of the paper table. |
| reported | the **best test epoch** of each run, as mean ± sd over seeds, and the EP-KD − KD difference paired by seed |

**Kaggle settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**. Then use **Save Version → Save & Run All
(Commit)**, so that the run continues after you close the browser.

**Resuming.** Every run saves a resumable state (`latest.pt`, after each epoch) under `/kaggle/working/runs/`. The
runner has no time limit of its own; Kaggle ends a session after 12 h. If a version ends with unfinished runs and its
output was saved:

1. Open the notebook in the editor and choose **Add Input → Your Work → Notebooks**. Select this notebook, so that its
   latest output version is attached.
2. Run **Save & Run All** again. The restore cell finds the previous `runs/` folder under `/kaggle/input`, and each run
   continues from its last finished epoch.

The jobs run concurrently: `JOBS_PER_GPU` jobs on each GPU. The EP-KD jobs start first; with `RUN_KD = True` the KD
jobs fill the free slots.

In [ ]:
# ---- configuration ----
METHOD = "vrm_epkd"       # epkd, crld_epkd or vrm_epkd (one method per notebook)
TEACHER, STUDENT = "ResNet50", "MobileNetV2"
TEACHER_ACC = 79.34      # test accuracy of the teacher checkpoint (sanity check only)
LR = 0.01                 # base learning rate (CRD / DKD setting for this student)
SEEDS = [0, 1, 2]
RUN_KD = False            # EP-KD notebook only: also train KD (ours), standard schedule, same seeds (table row + Delta)
JOBS_PER_GPU = 2          # concurrent training processes per GPU (data and augmentation live on the GPU)
REPO = "https://github.com/Soobiwan/epkd-vision.git"
REPO_REF = "main"         # branch, tag or commit of epkd-vision
EXTRA_ARGS = []          # extra train.py flags for every run, e.g. ["--epochs", "2"] for a smoke test

PAIR = f"{TEACHER}_to_{STUDENT}"
CODE = "/kaggle/temp/epkd-vision"   # clone, data and teacher (not part of the saved output)
OUT = "/kaggle/working/runs"         # run folders (saved as the notebook output)
import time; T_START = time.time()

In [ ]:
import os, subprocess, sys
os.makedirs("/kaggle/temp", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", REPO, CODE], check=True)
subprocess.run(["git", "-C", CODE, "fetch", "--all", "--tags", "-q"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", REPO_REF], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "--ff-only"], check=False)  # no-op on a detached commit
COMMIT = subprocess.run(["git", "-C", CODE, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import torch
N_GPU = torch.cuda.device_count()
print(f"epkd-vision @ {COMMIT}; torch {torch.__version__}; {N_GPU} GPU(s)")
assert N_GPU >= 1, "no GPU: set Accelerator to GPU T4 x2"
if N_GPU < 2:
    print("WARNING: only one GPU visible; the jobs will share it (choose 'GPU T4 x2' for both GPUs)")

In [ ]:
# CIFAR-100 (python version): an attached Kaggle copy is used if present, otherwise the official download
import glob, shutil
data = os.path.join(CODE, "data")
os.makedirs(data, exist_ok=True)
if not os.path.isdir(os.path.join(data, "cifar-100-python")):
    found = [p for p in glob.glob("/kaggle/input/**/cifar-100-python", recursive=True)
             if os.path.exists(os.path.join(p, "train"))]
    if found:
        shutil.copytree(found[0], os.path.join(data, "cifar-100-python"))
        print("CIFAR-100 copied from", found[0])
    else:
        tgz = os.path.join(data, "cifar-100-python.tar.gz")
        subprocess.run(["curl", "-sSL", "--retry", "5", "-o", tgz,
                        "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"], check=True)
        subprocess.run(["tar", "-xzf", tgz, "-C", data], check=True)
        os.remove(tgz)
print(sorted(os.listdir(os.path.join(data, "cifar-100-python"))))

In [ ]:
# Pretrained teacher: RepDistiller (CRD) checkpoint, mdistiller mirror as fallback -> weights-only state dict
subprocess.run([sys.executable, "scripts/get_cifar_teachers.py", "--out", "teachers", TEACHER], cwd=CODE, check=True)
TEACHER_CKPT = os.path.join(CODE, "teachers", f"{TEACHER}_cifar100.pth")
print(TEACHER_CKPT, os.path.getsize(TEACHER_CKPT) // 1024, "KB")

In [ ]:
# Resume: copy run folders from a previous version of this notebook (attached via Add Input), if any
import json

def progress(run_dir):
    """(done, last epoch logged) of a run folder."""
    if os.path.exists(os.path.join(run_dir, "done.json")):
        return True, 10 ** 6
    ep = 0
    log = os.path.join(run_dir, "train.jsonl")
    if os.path.exists(log) and os.path.exists(os.path.join(run_dir, "latest.pt")):
        for line in open(log):
            try:
                ep = max(ep, json.loads(line)["epoch"])
            except Exception:
                pass
    return False, ep

os.makedirs(OUT, exist_ok=True)
for prev in glob.glob(f"/kaggle/input/**/runs/{PAIR}_*", recursive=True):
    if not os.path.isdir(prev):
        continue
    dst = os.path.join(OUT, os.path.basename(prev))
    if progress(prev)[1] > progress(dst)[1]:
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(prev, dst)
        print(f"restored {os.path.basename(prev)} (epoch {progress(dst)[1]}, done={progress(dst)[0]}) from {prev}")

In [ ]:
# Job runner: EP-KD (and KD) for every seed, JOBS_PER_GPU concurrent jobs per GPU
COMMON = ["--teacher", TEACHER, "--student", STUDENT, "--teacher-ckpt", TEACHER_CKPT,
          "--data-root", data, "--lr", str(LR)]
METHODS = {METHOD: ['--views', '1', '--view-cache', '1', '--view-set', 'sal1,sal2,sal3', '--view-rotate', '1', '--passes', '3,3,3,1,1', '--anchor', '1', '--milestones', '48,144,192,210', '--method', 'vrm', '--bs', '64'], "kd": ["--views", "0", "--milestones", "150,180,210"]}
RUN_METHODS = [METHOD] + (["kd"] if RUN_KD and METHOD == "epkd" else [])
jobs = [(m, s) for m in RUN_METHODS for s in SEEDS]  # the method first
pending = [j for j in jobs if not progress(os.path.join(OUT, f"{PAIR}_{j[0]}_s{j[1]}"))[0]]
print(f"{len(jobs) - len(pending)} done, {len(pending)} to run: {pending}")

def last_line(run_dir):
    try:
        return json.loads(open(os.path.join(run_dir, "train.jsonl")).read().strip().splitlines()[-1])
    except Exception:
        return {}

running = {}  # name -> (Popen, gpu, log file)
last_report = 0
while pending or running:
    for name, (p, g, fh) in list(running.items()):
        if p.poll() is not None:
            fh.close()
            del running[name]
            r = last_line(os.path.join(OUT, name))
            print(f"[{time.strftime('%H:%M')}] {name} exited ({p.returncode}); epoch {r.get('epoch')} best {r.get('best')}", flush=True)
    load = [sum(g == k for _, g, _ in running.values()) for k in range(N_GPU)]
    while pending and min(load) < JOBS_PER_GPU:
        g = load.index(min(load))
        m, s = pending.pop(0)
        name = f"{PAIR}_{m}_s{s}"
        run_dir = os.path.join(OUT, name)
        os.makedirs(run_dir, exist_ok=True)
        fh = open(os.path.join(run_dir, "stdout.log"), "a")
        cmd = [sys.executable, "train.py", *COMMON, *METHODS[m], *EXTRA_ARGS, "--seed", str(s), "--run-dir", run_dir]
        p = subprocess.Popen(cmd, cwd=CODE, stdout=fh, stderr=subprocess.STDOUT,
                             env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)})
        running[name] = (p, g, fh)
        load[g] += 1
        print(f"[{time.strftime('%H:%M')}] started {name} on GPU {g}: {' '.join(cmd[1:])}", flush=True)
    if time.time() - last_report > 900:
        last_report = time.time()
        el = (time.time() - T_START) / 3600
        print(f"[{time.strftime('%H:%M')}] {el:.1f} h elapsed", flush=True)
        for name in running:
            r = last_line(os.path.join(OUT, name))
            print(f"   {name}: epoch {r.get('epoch', 0)}/240  test {r.get('test_acc')}  best {r.get('best')}  "
                  f"{r.get('sec', 0):.0f} s/epoch", flush=True)
    time.sleep(30)
print("runner finished")

In [ ]:
# Summary: best test epoch per run, mean ± sd over seeds, EP-KD - KD paired by seed (95% t-CI)
import numpy as np
T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776}

def best_of(run_dir):
    best = None
    try:
        for line in open(os.path.join(run_dir, "train.jsonl")):
            r = json.loads(line)
            best = r["test_acc"] if best is None else max(best, r["test_acc"])
    except FileNotFoundError:
        pass
    return best

res, teacher_acc = {}, None
for m in RUN_METHODS:
    for s in SEEDS:
        rd = os.path.join(OUT, f"{PAIR}_{m}_s{s}")
        if os.path.isdir(rd):
            done, ep = progress(rd)
            res[(m, s)] = dict(best=best_of(rd), done=done, epoch=ep)
            try:
                teacher_acc = json.load(open(os.path.join(rd, "config.json")))["teacher_acc"]
            except Exception:
                pass
print(f"{PAIR}: teacher accuracy measured {teacher_acc} (expected {TEACHER_ACC})")
lines = [f"## {PAIR} (epkd-vision @ {COMMIT[:7]})", "", "| method | " + " | ".join(f"s{s}" for s in SEEDS) + " | mean ± sd |",
         "|---|" + "---|" * (len(SEEDS) + 1)]
summary = dict(pair=PAIR, teacher=TEACHER, student=STUDENT, lr=LR, commit=COMMIT, teacher_acc=teacher_acc, runs={})
for m in RUN_METHODS:
    cells, vals = [], []
    for s in SEEDS:
        r = res.get((m, s))
        if r is None or r["best"] is None:
            cells.append("-")
            continue
        summary["runs"][f"{m}_s{s}"] = r
        cells.append(f"{r['best']:.2f}" + ("" if r["done"] else f" (ep {r['epoch']}, running)"))
        if r["done"]:
            vals.append(r["best"])
    ms = f"{np.mean(vals):.2f} ± {np.std(vals, ddof=1):.2f} (n={len(vals)})" if len(vals) > 1 else "-"
    summary[m] = dict(mean=float(np.mean(vals)) if vals else None,
                      sd=float(np.std(vals, ddof=1)) if len(vals) > 1 else None, n=len(vals))
    lines.append(f"| {m} | " + " | ".join(cells) + f" | {ms} |")
paired = [res[(METHOD, s)]["best"] - res[("kd", s)]["best"] for s in SEEDS
          if res.get((METHOD, s), {}).get("done") and res.get(("kd", s), {}).get("done")]
if len(paired) > 1:
    d, sd = float(np.mean(paired)), float(np.std(paired, ddof=1))
    h = T95[len(paired) - 1] * sd / np.sqrt(len(paired))
    summary["delta"] = dict(mean=d, ci95=[d - h, d + h], n=len(paired))
    lines += ["", f"EP-KD − KD (paired, n={len(paired)}): {d:+.2f} [{d - h:+.2f}, {d + h:+.2f}]"]
unfinished = [f"{m}_s{s}" for m in RUN_METHODS for s in SEEDS if not res.get((m, s), {}).get("done")]
if unfinished:
    lines += ["", f"UNFINISHED: {unfinished}. Attach this version's output and run again (see the top cell)."]
text = "\n".join(lines)
print(text)
open(f"/kaggle/working/summary_{METHOD}_{PAIR}.md", "w").write(text + "\n")
json.dump(summary, open(f"/kaggle/working/summary_{METHOD}_{PAIR}.json", "w"), indent=1)